In [1]:
import threading
import time
from pathlib import Path

import torch
import numpy as np
import tqdm

from diff_mfld.geodesic.geodesic_funcs import ExpMethod, LogMethod
from diff_mfld.geometry.metric import MetricField
from diff_mfld.mfld import ComputeMfld
from optim.constr_methods import ConstrSolverMethod
from optim.constrained.ralm import RalmCfg
from optim.constrained.rsqo import RsqoCfg
from optim.subsolver_methods import SubsolverMethod
from optim.subsolvers.rgd import RiemGradDescentCfg
from optim.subsolvers.rtr import RiemTrustRegionCfg
from problem import create_problem
from testing.testing_metrics import euclid_metric, scaled_metric, coupled_metric, asymmetric_metric

In [2]:
methods = [
    # ((ExpMethod.IVP, LogMethod.BVP), "ivpbvp"),
    ((ExpMethod.APPROX_O1, LogMethod.APPROX_O1), "o1"),
    ((ExpMethod.APPROX_O2, LogMethod.APPROX_O2), "o2"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O3), "o3"),
    ((ExpMethod.APPROX_O2, LogMethod.APPROX_O1), "o2_o1"),
    ((ExpMethod.APPROX_O3, LogMethod.APPROX_O1), "o3_o1"),
    ((ExpMethod.APPROX_O3, LogMethod.APPROX_O2), "o3_o2"),
]

# linear scaling of the domain
max_domain_scaling = [
    0.5, 1., 1.5, # 1.75
]

# parameters for random starting positions
p0_mean = np.array([2., -5., -8.])
p0_covar = 2. ** 2 * np.eye(3)
num_p0 = 20

# cost minimized at this point
cost_center = np.array([-5., -1., 3.])
cntr_center = np.array([-2., 3., 1.])
cntr_radius = 3.


# metrics to use in testing

metric_funcs = [
    # (euclid_metric, "euclid"),
    # (scaled_metric, "scaled"),
    (coupled_metric, "coupled"),
    # (asymmetric_metric, "asymmetric"),
]


In [3]:
# generates the random points and scales them

# "To Everything? To the great Question of Life, the Universe and everything?"
r = np.random.default_rng(42)
p_rand_starting = r.multivariate_normal(p0_mean, p0_covar, num_p0)
max_starting_norm = np.max(np.linalg.vector_norm(p_rand_starting, axis=1))

# scales the optimization functions and starting points
p_rand_starting /= max_starting_norm
cost_center /= max_starting_norm
cntr_center /= max_starting_norm
cntr_radius /= max_starting_norm

In [4]:
# configure the constrained configurations
rsqo_cfg = RsqoCfg()

optimizers = [
    ((ConstrSolverMethod.RSQO, rsqo_cfg), "rsqo"),
]

In [5]:
# root directory to store output files inside
base_data_dir = Path("../data/constrained")

In [6]:
from diff_mfld.mfld import Mfld
import itertools

# run the optimization scheme

cfgs = itertools.product(
    metric_funcs, max_domain_scaling, optimizers, methods
)
cfg_pbar = tqdm.tqdm(cfgs, desc="Configurations")

for (
        (metric_fn, metric_fn_label),
        scaling,
        ((optimizer, optimizer_cfg), optimizer_label),
        ((exp_method, log_method), method_label)
) in cfg_pbar:

    # setup the manifold configuration
    metric = MetricField(lambda x1, x2, x3: metric_fn(x1, x2, x3, scaling=scaling))
    conn = metric.christoffels()  # Levi-Civita connection (metric-compatible and torsion-free)
    mfld = Mfld(metric, conn)

    compute_mfld = ComputeMfld(
        mfld=mfld,
        exp_method=exp_method,
        log_method=log_method,
        dist_method=log_method,  # which logarithm to use when computing distance
    )

    # scale the problem coordinates accordingly
    trials_p_starting = p_rand_starting * scaling
    trials_cost_center = cost_center * scaling

    trials_region_center = cntr_center * scaling
    trials_region_radius = cntr_radius * scaling

    # create the problem
    cost, region_ineqs = create_problem(torch.tensor(trials_cost_center), [[(torch.tensor(trials_region_center), trials_region_radius)]])
    region_ineq = region_ineqs[0]

    trials_pbar = tqdm.tqdm(range(trials_p_starting.shape[0]), desc="Trials")
    for start_idx in trials_pbar:
        p0 = torch.tensor(trials_p_starting[start_idx])
        result = optimizer(cost, [region_ineq], [], p0, compute_mfld, optimizer_cfg)
        # print(result)

        # save the resulting dataclass
        filename = f"{optimizer_label}/metric_{metric_fn_label}__scaling_{scaling}__trial_{start_idx}__geod_method_{method_label}.pt"
        file_path = base_data_dir / filename
        print(f"\tSaving to {file_path}")

        torch.save(result, file_path)

Configurations: 0it [00:00, ?it/s]
Trials:  10%|█         | 2/20 [00:00<00:02,  6.79it/s]

	Saving to ../data/constrained/rsqo/metric_coupled__scaling_0.5__trial_0__geod_method_o1.pt
	Saving to ../data/constrained/rsqo/metric_coupled__scaling_0.5__trial_1__geod_method_o1.pt



Trials:  15%|█▌        | 3/20 [00:00<00:02,  6.80it/s]

	Saving to ../data/constrained/rsqo/metric_coupled__scaling_0.5__trial_2__geod_method_o1.pt
	Saving to ../data/constrained/rsqo/metric_coupled__scaling_0.5__trial_3__geod_method_o1.pt



Trials:  30%|███       | 6/20 [00:00<00:01,  7.62it/s]

...

	Saving to ../data/constrained/rsqo/metric_coupled__scaling_1.5__trial_17__geod_method_o3_o2.pt



Trials:  95%|█████████▌| 19/20 [38:58<00:51, 51.71s/it]

	Saving to ../data/constrained/rsqo/metric_coupled__scaling_1.5__trial_18__geod_method_o3_o2.pt



Trials: 100%|██████████| 20/20 [41:53<00:00, 125.69s/it][A
Configurations: 15it [2:29:46, 599.10s/it] 

	Saving to ../data/constrained/rsqo/metric_coupled__scaling_1.5__trial_19__geod_method_o3_o2.pt
